In [ ]:
import face_recognition
import os
import pickle
import cv2
import numpy as np
from tqdm import tqdm

dataset_path = r'C:\Users\Precision\Desktop\eng\cap\data\LFW_without_Mask'
output_file = "trained_faces_data.pkl"

known_face_encodings = []
known_face_names = []

print("🚀 محاولة التدريب مع إصلاح تنسيق الذاكرة (Memory Layout)...")

person_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]

for person_name in tqdm(person_folders, desc="Processing People"):
    person_dir = os.path.join(dataset_path, person_name)
    
    for image_name in os.listdir(person_dir):
        if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(person_dir, image_name)
            
            try:
                img = cv2.imread(image_path)
                if img is None: continue

                # 1. تحويل لـ RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # 2. الحل السحري: إجبار المصفوفة أن تكون متصلة وفي الذاكرة بنظام C-style
                # وتحويلها يدوياً لـ uint8 لضمان عدم وجود أي Float64 خفي
                img_final = np.ascontiguousarray(img_rgb, dtype=np.uint8)
                
                # 3. محاولة استخراج البصمة
                encodings = face_recognition.face_encodings(img_final)
                
                if len(encodings) > 0:
                    known_face_encodings.append(encodings[0])
                    known_face_names.append(person_name)
                    
            except Exception:
                continue

if len(known_face_encodings) > 0:
    data = {"encodings": known_face_encodings, "names": known_face_names}
    with open(output_file, "wb") as f:
        pickle.dump(data, f)
    print(f"\n✅ نجحنا أخيراً! تم حفظ {len(known_face_encodings)} وجه.")
else:
    print("\n❌ لا تزال dlib ترفض الصور. يرجى التحقق من إصدار NumPy.")

In [2]:
import pickle

# تحميل الملف الذي تم إنشاؤه
with open("trained_faces_data.pkl", "rb") as f:
    data = pickle.load(f)

print(f"Total Names Loaded: {len(data['names'])}")
print(f"Total Encodings Loaded: {len(data['encodings'])}")
print(f"First 5 People in Database: {list(set(data['names']))[:5]}")

Total Names Loaded: 2106
Total Encodings Loaded: 2106
First 5 People in Database: ['Luis_Ernesto_Derbez_Bautista', 'Kurt_Warner', 'Jiri_Novak', 'Colin_Montgomerie', 'Guillermo_Coria']


In [1]:
import face_recognition
import cv2
import pickle
import numpy as np

# 1. تحميل قاعدة البيانات التي تعبنا في إنشائها
print("⌛ جاري تحميل قاعدة بيانات الوجوه...")
with open("trained_faces_data.pkl", "rb") as f:
    data = pickle.load(f)

# 2. فتح الكاميرا (رقم 0 هو الكاميرا الافتراضية)
video_capture = cv2.VideoCapture(0)

print("🚀 الكاميرا تعمل الآن.. اضغط 'q' للخروج")

while True:
    # التقاط إطار من الكاميرا
    ret, frame = video_capture.read()
    if not ret:
        break

    # تصغير الصورة قليلاً لتسريع عملية المعالجة (اختياري)
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    
    # تحويل الصورة من BGR إلى RGB
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # تحديد أماكن الوجوه واستخراج البصمات
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        # المقارنة مع قاعدة البيانات
        matches = face_recognition.compare_faces(data["encodings"], face_encoding, tolerance=0.5)
        name = "Unknown"

        # استخدام أقرب مسافة لزيادة الدقة
        face_distances = face_recognition.face_distance(data["encodings"], face_encoding)
        best_match_index = np.argmin(face_distances)
        if matches[best_match_index]:
            name = data["names"][best_match_index]

        face_names.append(name)

    # عرض النتائج على الشاشة
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        # إعادة تكبير الإحداثيات لأننا صغرنا الصورة في البداية
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        # رسم مربع حول الوجه
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)

        # وضع التسمية بالاسم
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 255, 0), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.6, (255, 255, 255), 1)

    # إظهار النافذة
    cv2.imshow('GuardIQ - Face Recognition Test', frame)

    # الخروج عند الضغط على حرف 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# إغلاق كل شيء
video_capture.release()
cv2.destroyAllWindows()

C:\Users\Precision\anaconda3\envs\face_env\lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


⌛ جاري تحميل قاعدة بيانات الوجوه...
🚀 الكاميرا تعمل الآن.. اضغط 'q' للخروج


KeyboardInterrupt: 